In [2]:
import pandas as pd

df = pd.read_csv('../data/processed/inventory_clean.csv')

df['date'] = pd.to_datetime(df['date'])

print(df.describe())

                      date  current_stock  daily_demand  lead_time_days  \
count                 2800    2800.000000   2800.000000     2800.000000   
mean   2027-10-31 12:00:00     250.288214     61.925714        7.432143   
min    2024-01-01 00:00:00       0.000000      5.000000        1.000000   
25%    2025-11-30 18:00:00     125.000000     33.000000        4.000000   
50%    2027-10-31 12:00:00     249.000000     62.000000        7.000000   
75%    2029-09-30 06:00:00     376.250000     90.000000       11.000000   
max    2031-08-31 00:00:00     499.000000    119.000000       14.000000   
std                    NaN     143.466643     32.981750        4.006029   

       supplier_reliability_score  
count                 2800.000000  
mean                    79.252857  
min                     60.000000  
25%                     70.000000  
50%                     79.000000  
75%                     89.000000  
max                     99.000000  
std                     11.235899  


In [3]:
df['stockout_risk'].value_counts()

print("\n--- Demanda media según riesgo ---")
print(df.groupby('stockout_risk')['daily_demand'].mean())


--- Demanda media según riesgo ---
stockout_risk
No     61.711374
Yes    62.581159
Name: daily_demand, dtype: float64


La demanda por sí sola no predice la rotura de stock. Si el modelo solo mirara la cantidad demandada, no sabría distinguir entre un caso de riesgo y uno que no lo es. El modelo va a necesitar aprender de la combinación de todas las variables (stock actual, fiabilidad del proveedor, promociones, etc.).

In [4]:
# Convertir columnas binarias
df['stockout_risk'] = df['stockout_risk'].map({'No': 0, 'Yes': 1})
df['promotion_active'] = df['promotion_active'].map({'No': 0, 'Yes': 1})

# Convertir variables con más categorías (como clima) a columnas separadas
df = pd.get_dummies(df, columns=['weather_impact'], drop_first=True)

In [ ]:
# Ver si hay celdas vacías
print(df.isnull().sum())

# Si hay nulos, o los borramos o los rellenamos si hay muchos. Los borramos.
df = df.dropna()

product_id                    0
date                          0
store_id                      0
current_stock                 0
daily_demand                  0
lead_time_days                0
supplier_reliability_score    0
promotion_active              0
stockout_risk                 0
weather_impact_Low            0
weather_impact_Medium         0
dtype: int64


In [6]:
# Eliminar filas idénticas
df = df.drop_duplicates()

In [ ]:
# Quitamos las columnas que no aportan información predictiva al modelo
df_model = df.drop(columns=['date', 'product_id', 'store_id'])

# Guardamos el archivo final
df_model.to_csv('../data/processed/inventory_ready_for_model.csv', index=False)
print("Archivo limpio en data/processed/inventory_ready_for_model.csv")

¡Archivo listo en data/processed/inventory_ready_for_model.csv!
